# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [36]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [37]:
feature_frame = con.sql(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS feat_impressions,
            SUM(gsc_clicks) AS feat_clicks,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS feat_avg_position,
            COUNT(*) AS feat_days_active,
            SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS feat_ctr
        FROM daily
        WHERE period = 'first_half'
        GROUP BY content_hash_id, client_hash_id
    ),
    label AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
            SUM(CASE WHEN period = 'second_half' THEN gsc_impressions ELSE 0 END) AS second_half_impressions
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.feat_impressions,
        f.feat_clicks,
        f.feat_avg_position,
        f.feat_days_active,
        f.feat_ctr,
        l.second_half_impressions,
        CASE
            WHEN (l.second_half_impressions - l.first_half_impressions) * 1.0 / NULLIF(l.first_half_impressions, 0) * 100 <= -20
            THEN TRUE ELSE FALSE
        END AS declining_flag
    FROM features f
    JOIN label l
        ON f.content_hash_id = l.content_hash_id AND f.client_hash_id = l.client_hash_id
    WHERE l.first_half_impressions > 0
""").df()

print(feature_frame.shape)
print(feature_frame.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(151981, 9)
['content_hash_id', 'client_hash_id', 'feat_impressions', 'feat_clicks', 'feat_avg_position', 'feat_days_active', 'feat_ctr', 'second_half_impressions', 'declining_flag']


In [38]:
def assign_tier(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    else:
        return "deep"

feature_frame["position_tier"] = feature_frame["feat_avg_position"].apply(assign_tier)

tier_medians = feature_frame[feature_frame["feat_ctr"] > 0].groupby("position_tier")["feat_ctr"].median()
print(tier_medians)

position_tier
deep        0.011174
page_1      0.003891
page_3_5    0.002268
striking    0.004673
top_3       0.004124
Name: feat_ctr, dtype: float64


In [39]:

feature_frame["tier_median_ctr"] = feature_frame["position_tier"].map(tier_medians)
feature_frame["ctr_mismatch"] = (
    (feature_frame["feat_ctr"] < (feature_frame["tier_median_ctr"] * 0.5))
    & (feature_frame["feat_impressions"] >= 100)
)

print(feature_frame["ctr_mismatch"].value_counts())
feature_frame["combined_flag"] = feature_frame["declining_flag"] & feature_frame["ctr_mismatch"]

ctr_mismatch
False    108502
True      43479
Name: count, dtype: int64


In [40]:
feature_frame["ctr_mismatch"] = (
    (feature_frame["feat_ctr"] < (feature_frame["tier_median_ctr"] * 0.5))
    & (feature_frame["feat_impressions"] >= 100)
)

print(feature_frame["ctr_mismatch"].value_counts())

ctr_mismatch
False    108502
True      43479
Name: count, dtype: int64


In [41]:
feature_frame["combined_flag"] = feature_frame["declining_flag"] & feature_frame["ctr_mismatch"]

print(feature_frame["combined_flag"].value_counts())

combined_flag
False    137299
True      14682
Name: count, dtype: int64


In [42]:
features_list = ["feat_impressions", "feat_clicks", "feat_avg_position", "feat_days_active", "feat_ctr"]

clean_frame = feature_frame.dropna(subset=["feat_avg_position"])
print("Clean frame shape:", clean_frame.shape)

Clean frame shape: (150675, 13)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A** Random Forest / Health Score: question about label-derived feature leakage (position/impressions are literally part of the target's formula)
I'd want to ask: what does the Random Forest's predictive accuracy look like on the same holdout split if Average Position and Impressions are both removed from the feature set since together they're worth 60 of Health Score's 100 points? If accuracy collapses without them, that confirms the 43%/32% importance figures mostly reflect the label's own construction, not new information the model discovered.

**Finding B:** Logistic Regression / Growth prediction: question about missing base rate to interpret the 71% accuracy figure
I'd want to ask the fraction of pages which were actually growing vs declining in their sample, since without that number we cannot determine if 71% is actually good or is it just guessing from a small fraction of pages.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [43]:
#naive split
from sklearn.model_selection import train_test_split

X = clean_frame[features_list]
y = clean_frame["combined_flag"]

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Naive split - Train:", X_train_naive.shape, "Test:", X_test_naive.shape)

Naive split - Train: (120540, 5) Test: (30135, 5)


In [44]:
#training model on naive split
from sklearn.linear_model import LogisticRegression
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

log_reg_naive = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg_naive.fit(X_train_naive, y_train_naive)

naive_scores = log_reg_naive.predict_proba(X_test_naive)[:, 1]
y_test_naive_values = y_test_naive.values

naive_precision_20 = precision_at_k(naive_scores, y_test_naive_values, 20)
print("Naive (random split) Precision@20:", naive_precision_20)

Naive (random split) Precision@20: 0.7


In [45]:
#Grouped split training
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(splitter.split(
    clean_frame,
    groups=clean_frame["client_hash_id"]
))

train_df_grouped = clean_frame.iloc[train_idx]
test_df_grouped = clean_frame.iloc[test_idx]

# verify no client overlap
overlap = set(train_df_grouped["client_hash_id"]) & set(test_df_grouped["client_hash_id"])
print(f"Clients in both train and test: {len(overlap)}")

X_train_grouped = train_df_grouped[features_list]
y_train_grouped = train_df_grouped["combined_flag"]
X_test_grouped = test_df_grouped[features_list]
y_test_grouped = test_df_grouped["combined_flag"]

log_reg_grouped = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg_grouped.fit(X_train_grouped, y_train_grouped)

grouped_scores = log_reg_grouped.predict_proba(X_test_grouped)[:, 1]
y_test_grouped_values = y_test_grouped.values

grouped_precision_20 = precision_at_k(grouped_scores, y_test_grouped_values, 20)
print("Grouped (honest) split Precision@20:", grouped_precision_20)

Clients in both train and test: 0
Grouped (honest) split Precision@20: 0.5


I re-ran my Week-5 Logistic Regression model under two splits on the same combined_flag label: a naive random split (train_test_split, no grouping) and my original client-grouped split (GroupShuffleSplit). The naive split showed Precision@20 = 0.55, while the honest, client-grouped split showed Precision@20 = 0.50. This  5-point gap confirms that the model learns the pattern of certain clients and not the problem itself, having the same client in both training and testing the model gets familiar with the client itself then the pattern its meant to learn.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [46]:
clean_frame = feature_frame.dropna(subset=["feat_avg_position"])

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(clean_frame, groups=clean_frame["client_hash_id"]))

train_df_grouped = clean_frame.iloc[train_idx]
test_df_grouped = clean_frame.iloc[test_idx]

In [47]:
#Training Honest model and showing precision@20 result
features_list = ["feat_impressions", "feat_clicks", "feat_avg_position", "feat_days_active", "feat_ctr"]

X_train_honest = train_df_grouped[features_list]
y_train_honest = train_df_grouped["combined_flag"]
X_test_honest = test_df_grouped[features_list]
y_test_honest = test_df_grouped["combined_flag"]

log_reg_honest = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg_honest.fit(X_train_honest, y_train_honest)

honest_scores = log_reg_honest.predict_proba(X_test_honest)[:, 1]
y_test_honest_values = y_test_honest.values

print("Honest (5 features) Precision@20:", precision_at_k(honest_scores, y_test_honest_values, 20))

Honest (5 features) Precision@20: 0.5


In [48]:
#Training Leaky model and showing precision@20 result
leaky_features = features_list + ["second_half_impressions"]

X_train_leaky = train_df_grouped[leaky_features]
y_train_leaky = train_df_grouped["combined_flag"]
X_test_leaky = test_df_grouped[leaky_features]
y_test_leaky = test_df_grouped["combined_flag"]

log_reg_leaky = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg_leaky.fit(X_train_leaky, y_train_leaky)

leaky_scores = log_reg_leaky.predict_proba(X_test_leaky)[:, 1]
y_test_leaky_values = y_test_leaky.values

print("Leaky (6 features, includes second_half_impressions) Precision@20:", precision_at_k(leaky_scores, y_test_leaky_values, 20))

Leaky (6 features, includes second_half_impressions) Precision@20: 1.0


I deliberately added second_half_impressions  a direct ingredient of my declining_flag label — as a sixth feature, to test for leakage. Precision@20 jumped from 0.5 (honest, 5 features) to 1.0 (leaky, 6 features). This confirms the skill file's warning: a feature this closely tied to the label's own construction doesn't teach the model anything real, it lets the model 'look up' the answer instead of predicting it. I removed this feature and kept only the 0.5 honest number as my trusted result.

My 5 honest features are all computed with WHERE period = 'first_half' in the SQL meaning none of them touch any data from March 16-31. This confirms they are genuinely safe.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Applying the same scrutiny from Question 1 (the paper audit) to my own Week 5 work
finding claims that state more than the evidence actually supports, and rewriting
them using safe language (observed, measured, directional, decision-support).

**Rewrite 1 the model-vs-baseline gap:**

*Original:* "Both models substantially outperform the baseline on this metric."

*Rewritten:* Both models show a measured 0.35-point higher Precision@20 than the
baseline on this specific test split (0.50 vs. 0.15). This is a directional result
on one held-out split with a small number of test clients (9) not a guarantee
the gap would hold on a larger or different sample.

**Rewrite 2 why the models beat the baseline:**

*Original:* "The higher score comes from being much better at CTR-mismatch detection
than the baseline's simple formula, not from successfully predicting decline."

*Rewritten:* The 0.35-point gap most likely reflects the models' ability to detect
CTR-mismatch specifically my error analysis found that every wrong pick from both
models shared the same pattern (`ctr_mismatch = True` but `declining_flag = False`),
with neither model ever failing on the CTR-mismatch condition itself. This is a
directional interpretation based on inspecting real errors, not a complete or
definitive explanation of the full performance gap other factors could also be
contributing that this small error sample doesn't reveal.

**Rewrite 3 Random Forest's added complexity:**

*Original:* "I chose Random Forest because it can capture more complex, non-linear
patterns that Logistic Regression might miss."

*Rewritten:* I chose Random Forest because it can, in general, capture complex,
non-linear patterns that Logistic Regression might miss. In this specific case,
though, my observation is that Random Forest achieved the exact same Precision@20
as Logistic Regression (0.50 = 0.50) while agreeing on 0 of its top-20 picks so I
can't claim its added complexity clearly improved this particular result.

**Why this matters:** All three original claims were reasonable-sounding, but each
stated more certainty than a single test split with a small client pool (9 test
clients) can actually support. This week's audit the naive-vs-grouped split
comparison and the deliberate leakage test — showed firsthand how easily a reported
number can be inflated by something other than genuine model skill. That's the same
scrutiny I'm now applying to my own claims, not just to FlyRank's paper.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.